# Portfolio Financing System Prototype (V2)
**Author**: Chris Hsieh (chrishsh@gmail.com)

This notebook is the prototype implementation of the **Automated Portfolio Financing System**. 
It perfectly matches the logic of the full-blown Python suite (`portfolio_financing/` package) but is presented in an interactive cell-by-cell format.

It contains:
1. **Inventory Manager**: Seeds multi-strat positions.
2. **Internalization Engine**: Nets gross exposures.
3. **Compliance Engine**: US Reg T/SHO and APAC SBL checks.
4. **Locate Engine**: PB simulator for borrow rates.
5. **Collateral Optimizer**: MILP solver for optimal pledging.
6. **Manual Trade Entry**: Blotter for Repo, TRS, Sell/Buyback.
7. **P&L Engine**: Actual/360 daily accruals.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.optimize import linprog
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

In [2]:

# ==========================================
# 1. Inventory & Market Data
# ==========================================
class InventoryManager:
    def __init__(self, tickers, strategies):
        self.tickers = tickers
        self.strategies = strategies
        self.prices = {}
        
    def fetch_market_data(self):
        for t in self.tickers:
            try:
                dat = yf.Ticker(t).history(period='1d')
                self.prices[t] = dat['Close'].iloc[-1] if not dat.empty else 100.0
            except:
                self.prices[t] = 100.0
                
    def generate_mock_inventory(self):
        records = []
        np.random.seed(42)
        for strat in self.strategies:
            for t in self.tickers:
                qty = np.random.randint(-10000, 10000)
                if qty != 0:
                    records.append({
                        'Strategy': strat, 'Ticker': t,
                        'AssetClass': 'FixedIncome' if t in ['AGG', 'TLT'] else 'Equity',
                        'Jurisdiction': 'APAC' if '.HK' in t or '.T' in t else 'US',
                        'Quantity': qty, 'MtM_Price': self.prices.get(t, 100.0),
                        'Notional': qty * self.prices.get(t, 100.0)
                    })
        return pd.DataFrame(records)

# ==========================================
# 2. Internalization Engine
# ==========================================
class InternalizationEngine:
    def __init__(self, ledger):
        self.ledger = ledger
        
    def calculate_net_exposure(self):
        net_pos = self.ledger.groupby(['Ticker', 'AssetClass', 'Jurisdiction']).agg({'Quantity': 'sum', 'Notional': 'sum'}).reset_index()
        gross = self.ledger['Notional'].abs().sum()
        net = net_pos['Notional'].abs().sum()
        ratio = 1 - (net / gross) if gross > 0 else 0
        return net_pos, ratio

# ==========================================
# 3 & 4. Locates & Compliance
# ==========================================
class SecurityLocateEngine:
    def __init__(self, tickers):
        self.rates = {t: np.random.uniform(0.0025, 0.01) for t in tickers}
        if 'AAPL' in self.rates: self.rates['AAPL'] = 0.065 # HTB
    def get_locate(self, ticker):
        return {'Locate_ID': np.random.randint(100000, 999999), 'Borrow_Fee_Rate': self.rates.get(ticker, 0.01), 'Secured': True}

class RegulatoryComplianceEngine:
    def __init__(self, locate_engine):
        self.locate_engine = locate_engine
    def validate_exposure(self, net_exposure_df):
        records = []
        for _, row in net_exposure_df.iterrows():
            record = row.to_dict()
            record['Compliance_Status'] = 'Passed'
            if row['Quantity'] < 0:
                if row['Jurisdiction'] == 'US':
                    loc = self.locate_engine.get_locate(row['Ticker'])
                    record['Locate_ID'] = loc['Locate_ID']
                    record['Borrow_Fee_Rate'] = loc['Borrow_Fee_Rate']
                elif row['Jurisdiction'] == 'APAC':
                    if '0700.HK' in row['Ticker']: # Mock failure
                        record['Compliance_Status'] = 'REJECTED: APAC Naked Short Ban'
                        records.append(record)
                        continue
                    record['Borrow_Fee_Rate'] = 0.02
            records.append(record)
        return pd.DataFrame(records)

# ==========================================
# 5. Collateral Optimizer
# ==========================================
class CollateralOptimizer:
    def __init__(self, inventory, margin_reqs):
        self.inventory = inventory
        self.margin_reqs = margin_reqs
        self.cost = {t: 0.05 if t in ['AGG', 'TLT'] else 0.01 for t in inventory.keys()}
        self.haircuts = {t: 0.02 if t in ['AGG', 'TLT'] else 0.10 for t in inventory.keys()}
    def optimize(self):
        tickers = list(self.inventory.keys())
        reqs = list(self.margin_reqs.keys())
        if not tickers or not reqs: return pd.DataFrame(), False
        c = np.zeros(len(tickers) * len(reqs))
        for i, t in enumerate(tickers):
            for j in range(len(reqs)): c[i * len(reqs) + j] = self.cost[t]
        A_ub, b_ub = np.zeros((len(tickers), len(tickers) * len(reqs))), np.zeros(len(tickers))
        for i in range(len(tickers)):
            for j in range(len(reqs)): A_ub[i, i * len(reqs) + j] = 1
            b_ub[i] = self.inventory[tickers[i]]
        A_req, b_req = np.zeros((len(reqs), len(tickers) * len(reqs))), np.zeros(len(reqs))
        for j in range(len(reqs)):
            for i in range(len(tickers)): A_req[j, i * len(reqs) + j] = -(1 - self.haircuts[tickers[i]])
            b_req[j] = -self.margin_reqs[reqs[j]]
        A_ub, b_ub = np.vstack((A_ub, A_req)), np.concatenate((b_ub, b_req))
        res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=(0, None), method='highs')
        allocs = []
        if res.success:
            x = res.x.reshape((len(tickers), len(reqs)))
            for i, t in enumerate(tickers):
                for j, r in enumerate(reqs):
                    if x[i,j] > 0.01: allocs.append({'Ticker': t, 'CP': r, 'Notional': x[i,j]})
        return pd.DataFrame(allocs), res.success

# ==========================================
# 6 & 7. Manual Trades & PnL
# ==========================================
class ManualTradeEntry:
    def __init__(self): self.trades = []
    def book_repo(self, t, p, r): self.trades.append({'Type': 'Repo', 'Ticker': t, 'Notional': p, 'Rate': r})
    def book_trs(self, t, q, p, d, r): self.trades.append({'Type': 'TRS', 'Ticker': t, 'Quantity': q, 'Notional': q*p, 'Direction': d, 'Rate': r})
    def get_blotter(self): return pd.DataFrame(self.trades)

class PnLEngine:
    def __init__(self, benchmark=0.05): self.bench = benchmark
    def calc_auto(self, df):
        if df.empty: return pd.DataFrame(), 0
        recs = []
        tot = 0
        for _, r in df.iterrows():
            if r['Compliance_Status'] != 'Passed': continue
            if r['Quantity'] < 0:
                c = -(abs(r['Notional']) * r.get('Borrow_Fee_Rate', 0.01)) / 360
                recs.append({'Ticker': r['Ticker'], 'PnL': c}); tot += c
            else:
                c = -(r['Notional'] * self.bench) / 360
                recs.append({'Ticker': r['Ticker'], 'PnL': c}); tot += c
        return pd.DataFrame(recs), tot


In [3]:
# Run Initialization
tickers = ['AAPL', 'MSFT', 'JPM', 'AGG', 'TLT', '0700.HK']
strats = ['StatArb', 'VolArb', 'Macro']
mgr = InventoryManager(tickers, strats)
mgr.fetch_market_data()
ledger = mgr.generate_mock_inventory()

int_eng = InternalizationEngine(ledger)
net_exp, ratio = int_eng.calculate_net_exposure()

loc_eng = SecurityLocateEngine(tickers)
comp_eng = RegulatoryComplianceEngine(loc_eng)
valid = comp_eng.validate_exposure(net_exp)

manual = ManualTradeEntry()
pnl = PnLEngine()

print(f"Initialization Complete! Internalization Ratio: {ratio*100:.2f}%")
print("\nValidated Exposures:")
display(valid)


Initialization Complete! Internalization Ratio: 32.98%

Validated Exposures:


,Ticker,AssetClass,Jurisdiction,Quantity,Notional,Compliance_Status,Locate_ID,Borrow_Fee_Rate
0,0700.HK,Equity,APAC,-8993,-4.021670e+06,REJECTED: APAC Naked Short Ban,NaN,NaN
1,AAPL,Equity,US,4330,1.353623e+06,Passed,NaN,NaN
2,AGG,FixedIncome,US,-1362,-1.330742e+05,Passed,249503.0,0.002673
3,JPM,Equity,US,-7754,-2.789812e+06,Passed,754811.0,0.002553
4,MSFT,Equity,US,-23945,-1.152353e+07,Passed,627035.0,0.007087
5,TLT,FixedIncome,US,2358,1.953131e+05,Passed,NaN,NaN
